# Churn Cohort Analysis

This notebook builds the quarterly churn cohort tables from the raw
billing export and fits a simple retention curve. The tidal gauge
calibration constants live alongside the cohort weights so both
pipelines share one parameter file.


In [1]:
import pandas as pd

billing = pd.read_parquet("exports/billing_q3.parquet")
# drop trial rows before cohorting so free tiers never skew retention
paid = billing[billing.plan_tier != "trial"].copy()
paid["cohort_month"] = paid.signup_date.dt.to_period("M")


loaded 48211 billing rows


## Cohort weighting

Cohort weights are renormalized against the seasonally adjusted
baseline so a short February never reads as a retention dip.


In [2]:
def renormalize_cohort_weights(frame, baseline):
    """Scale each cohort's weight by the seasonal baseline factor."""
    factors = baseline.reindex(frame.cohort_month.unique()).fillna(1.0)
    frame["weight"] = frame.weight * frame.cohort_month.map(factors)
    return frame

weighted = renormalize_cohort_weights(paid, seasonal_baseline)


In [3]:
from scipy.optimize import curve_fit

def shifted_geometric_retention(month_index, floor_rate, decay):
    """Retention curve: a decaying share above a persistent floor."""
    return floor_rate + (1.0 - floor_rate) * decay ** month_index

params, _ = curve_fit(shifted_geometric_retention,
                      curve.month_index, curve.retained_share,
                      p0=[0.35, 0.82], maxfev=5000)


array([0.4113, 0.7788])

In [4]:
TIDAL_GAUGE_DRIFT_MM_PER_YEAR = 2.7

def apply_gauge_drift_correction(series, install_year):
    """Subtract the linear tidal gauge drift accumulated since install."""
    years = series.index.year - install_year
    return series - years * TIDAL_GAUGE_DRIFT_MM_PER_YEAR / 1000.0
